# NexCart Intelligence — DistilBERT Sentiment Analysis
**Dataset:** Amazon Fine Food Reviews (568K reviews)
**Model:** DistilBertForSequenceClassification (fine-tuned)
**Target:** F1 > 0.85 on validation set

In [ ]:
import sys
print(sys.executable)

In [1]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("Libraries loaded ✅")

PyTorch version: 2.3.0+cpu
CUDA available: False
Libraries loaded ✅


## Step 1 — Load & Prepare Dataset

In [2]:
df = pd.read_csv('../data/Reviews.csv')
print(f"Full dataset shape: {df.shape}")
print(df[['Score', 'Text']].head(3))

Full dataset shape: (568454, 10)
   Score                                               Text
0      5  I have bought several of the Vitality canned d...
1      1  Product arrived labeled as Jumbo Salted Peanut...
2      4  This is a confection that has been around a fe...


In [3]:
# Keep only Score and Text columns
df = df[['Score', 'Text']].dropna()

# Binary classification: 1-3 → negative (0), 4-5 → positive (1)
df['label'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nPositive: {df['label'].sum()} | Negative: {(df['label']==0).sum()}")

Label distribution:
label
1    443777
0    124677
Name: count, dtype: int64

Positive: 443777 | Negative: 124677


In [4]:
# Separate positive and negative
positive = df[df['label'] == 1]
negative = df[df['label'] == 0]

print(f"Total positive: {len(positive)}")
print(f"Total negative: {len(negative)}")

# Balance: 50K each
sample_size = min(50_000, len(positive), len(negative))
positive_sample = positive.sample(n=sample_size, random_state=42)
negative_sample = negative.sample(n=sample_size, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([positive_sample, negative_sample])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print(f"Label distribution:\n{df_balanced['label'].value_counts()}")

Total positive: 443777
Total negative: 124677

Balanced dataset shape: (100000, 3)
Label distribution:
label
0    50000
1    50000
Name: count, dtype: int64


## Step 2 — Train / Validation / Test Split

In [5]:
# 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df_balanced, test_size=0.30, random_state=42, stratify=df_balanced['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print(f"Train size:      {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size:       {len(test_df)}")

Train size:      70000
Validation size: 15000
Test size:       15000


## Step 3 — Tokenization

In [6]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer loaded: {MODEL_NAME} ✅")

Tokenizer loaded: distilbert-base-uncased ✅


In [7]:
def tokenize(batch):
    return tokenizer(
        batch['Text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['Text', 'label']])
val_dataset   = Dataset.from_pandas(val_df[['Text', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['Text', 'label']])

# Apply tokenization
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete ✅")
print(f"Train dataset: {train_dataset}")

Map:   0%|          | 0/70000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenization complete ✅
Train dataset: Dataset({
    features: ['Text', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 70000
})


## Step 4 — Fine-Tune DistilBERT

In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f"Model loaded: {MODEL_NAME} ✅")
print(f"Parameters: {model.num_parameters():,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: distilbert-base-uncased ✅
Parameters: 66,955,010


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    accuracy = (predictions == labels).mean()
    return {
        "f1": round(f1, 4),
        "accuracy": round(accuracy, 4)
    }

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

print("Training arguments set ✅")

Training arguments set ✅


C:\Users\bomma\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning... (this takes 20-40 mins on CPU)")
trainer.train()
print("Fine-tuning complete ✅")

Starting fine-tuning... (this takes 20-40 mins on CPU)


Epoch,Training Loss,Validation Loss


## Step 5 — Evaluate on Validation Set

In [ ]:
results = trainer.evaluate(val_dataset)

print(f"Validation F1:       {results['eval_f1']}")
print(f"Validation Accuracy: {results['eval_accuracy']}")
print(f"\nTarget F1 > 0.85: {'✅ PASSED' if results['eval_f1'] > 0.85 else '❌ FAILED'}")

In [ ]:
# Get all predictions on test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print(classification_report(
    true_labels,
    pred_labels,
    target_names=['negative', 'positive']
))

## Step 6 — Save Model + Tokenizer

In [ ]:
os.makedirs('../saved_models/sentiment_model', exist_ok=True)

model.save_pretrained('../saved_models/sentiment_model')
tokenizer.save_pretrained('../saved_models/sentiment_model')

print("sentiment_model saved ✅")
print("tokenizer saved ✅")

## Step 7 — Quick Inference Test

In [ ]:
def predict_sentiment(text: str) -> dict:
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probs).item()
    confidence = probs[0][predicted_class].item()

    return {
        "label": "positive" if predicted_class == 1 else "negative",
        "confidence": round(confidence, 4)
    }

# Test positive review
result1 = predict_sentiment("This product is absolutely amazing, best purchase ever!")
print(f"Positive test: {result1}")

# Test negative review
result2 = predict_sentiment("Terrible quality, complete waste of money, do not buy.")
print(f"Negative test: {result2}")